# Heretic 2B Kaggle Proof

Runs upstream `heretic-llm` against an official base checkpoint, saves a merged checkpoint, and validates the files needed by the ONNX packaging pipeline.

Set Kaggle accelerator to **GPU T4 x2** and Internet to **On** before running.

In [ ]:
import os
from pathlib import Path

LABEL = os.environ.get("HERETIC_PROOF_LABEL", "alkahest-2b")  # alkahest-2b or rally-2b
REPO_URL = os.environ.get("HERETIC_TO_ONNX_REPO", "https://github.com/alkahest-ai/heretic-to-onnx.git")
REPO_REF = os.environ.get("HERETIC_TO_ONNX_REF", "codex/kaggle-heretic-2b-run")
WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "heretic-to-onnx"

print({"label": LABEL, "repo_url": REPO_URL, "repo_ref": REPO_REF, "repo_dir": str(REPO_DIR)})

In [ ]:
import torch, shutil, platform
from pathlib import Path

print("python_platform=", platform.platform())
print("cuda_available=", torch.cuda.is_available())
print("gpu_count=", torch.cuda.device_count())
gpu_names = []
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    gpu_names.append(p.name)
    print(f"gpu_{i}=", p.name, round(p.total_memory / 1024**3, 2), "GiB")
usage = shutil.disk_usage(Path("/kaggle/working"))
print("working_disk_free_gb=", round(usage.free / 1024**3, 2))

if os.environ.get("ALLOW_NON_T4X2", "0") != "1":
    if torch.cuda.device_count() != 2 or not all("T4" in name for name in gpu_names):
        raise RuntimeError(f"Expected Kaggle GPU T4 x2, got {gpu_names}. Change accelerator to GPU T4 x2 before running.")

In [ ]:
!pip install -U --no-cache-dir heretic-llm accelerate bitsandbytes peft datasets huggingface_hub hf_transfer
!pip install -U --no-cache-dir git+https://github.com/huggingface/transformers.git

In [ ]:
if not REPO_DIR.exists():
    !git clone --depth 1 --branch {REPO_REF} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {REPO_REF}
    !git checkout {REPO_REF}
    !git pull --ff-only
%cd {REPO_DIR}
!git rev-parse --short HEAD

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ.setdefault("HF_TOKEN", secrets.get_secret("HF_TOKEN"))
    try:
        os.environ.setdefault("HF_MERGED_REPO_ID", secrets.get_secret("HF_MERGED_REPO_ID"))
    except Exception:
        pass
except Exception:
    pass

if "HF_TOKEN" in os.environ:
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
    print("HF login ok")
else:
    print("HF_TOKEN not set; running without Hub upload")

In [ ]:
UPLOAD_ARGS = ""
os.environ.setdefault("HF_MERGED_REPO_ID", "thomasjvu/alkahest-2b-heretic-merged")
if os.environ.get("HF_TOKEN") and os.environ.get("HF_MERGED_REPO_ID"):
    UPLOAD_ARGS = f"--upload-merged-to {os.environ['HF_MERGED_REPO_ID']}"

!python scripts/kaggle_heretic_2b_proof.py \
  --label {LABEL} \
  --accelerator t4x2 \
  --n-trials 20 \
  --n-startup-trials 8 \
  --prompt-rows 160 \
  --eval-rows 80 \
  --max-response-length 64 \
  {UPLOAD_ARGS}

In [ ]:
!HERETIC_PROOF_LABEL={LABEL} python - <<'PY'
import json
import os
from pathlib import Path

label = os.environ.get('HERETIC_PROOF_LABEL', 'alkahest-2b')
merged_name = 'alkahest-2b-heretic-merged' if label == 'alkahest-2b' else 'rally-2b-heretic-merged'
path = Path('/kaggle/working/heretic-to-onnx/heretic') / label / merged_name
required = ['config.json', 'generation_config.json', 'tokenizer_config.json']
missing = [name for name in required if not (path / name).exists()]
tokenizers = [p.name for p in path.glob('tokenizer*')] + [p.name for p in path.glob('*.model')]
weights = [p.name for p in path.glob('*.safetensors')] + [p.name for p in path.glob('pytorch_model*.bin')]
report = {'path': str(path), 'missing': missing, 'tokenizers': tokenizers, 'weights': weights, 'ok': not missing and bool(tokenizers) and bool(weights)}
print(json.dumps(report, indent=2))
if not report['ok']:
    raise SystemExit(1)
PY